# 039 — Clasificación logística y umbrales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Regresión logística:** score lineal `z = β₀ + βᵀx` pasado por la sigmoide
`p̂ = σ(z) = 1/(1+e^(−z))`. El modelo es lineal en el log-odds: `log(p/(1−p)) = z`;
cada unidad de xⱼ multiplica los odds por `e^{βⱼ}`.

**Pérdida (entropía cruzada):** `L = −(1/n)Σ[y log p̂ + (1−y) log(1−p̂)]` — convexa,
castiga sin cota la confianza equivocada, gradiente `(p̂−y)·x`. Sin solución cerrada;
con datos separables diverge salvo regularización.

**Umbral por costos:** el modelo da p̂; la decisión usa
`t* = C_FP/(C_FP + C_FN)`. Mover t reparte errores (precision↔recall), no mejora el score.

**Calibración:** p̂ ≈ 0.7 debe corresponder a ~70 % de positivos reales; se diagnostica con
el diagrama de confiabilidad y se corrige con Platt o isotónica en validación.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) x=0: z=−2, p̂ = 1/(1+e²) ≈ 0.119. x=2.5: z=0, p̂ = 0.5. x=5: z=2,
p̂ ≈ 0.881. (b) p̂=0.5 cuando z=0, es decir x = 2/0.8 = 2.5. (c) Odds ratio por unidad:
e^0.8 ≈ 2.23 — cada unidad de x multiplica los odds por ~2.23.

**Ejercicio 2.** Modelo A: −(1/4)[log 0.9 + log 0.9 + log 0.8 + log 0.7] =
−(1/4)(−0.105 −0.105 −0.223 −0.357) ≈ **0.198**. Modelo B: −(1/4)[log 0.6·4 veces] =
−log 0.6 ≈ **0.511**. Misma accuracy, log-loss muy distinto: la entropía cruzada mide la
*calidad de la probabilidad* (confianza correcta), no solo el lado del umbral. A es
mejor score aunque ambos clasifiquen igual.

**Ejercicio 3.** (a) t* = 10/(10+490) = **0.02**. (b) Con t=0.5: p̂=0.05 < 0.5 → dejar
pasar; con t*=0.02: 0.05 ≥ 0.02 → revisar. La asimetría 49:1 de costos hace racional
revisar transacciones con apenas 5 % de riesgo. (c) Supone que p̂ está **calibrada**: si
0.05 no significa "5 % de fraudes reales", el umbral calculado no minimiza el costo.

**Ejercicio 4.** (a) La logística convierte la feature en un **score de probabilidad**
p̂ = σ(β₀ + β₁x): el umbral deja de vivir en las unidades de la feature y pasa a vivir en
probabilidad, donde se conecta con costos. (b) Con C_FN = 9·C_FP, el umbral óptimo es
t* = 1/(1+9) = 0.1 sobre p̂ **calibrada**; sobre la feature cruda equivale a resolver
σ(β₀ + β₁x) = 0.1 para x. El laboratorio, al maximizar accuracy, asume costos simétricos
(t = 0.5 implícito) — cambiar los costos cambia la decisión sin cambiar el modelo.


In [ ]:
result = run_lab("ml", seed=39)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — sigmoide y log-loss
import math

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

for x in (0, 2.5, 5):
    z = -2 + 0.8 * x
    print(f"x={x}: z={z:+.1f}  p̂={sigmoide(z):.3f}")

y   = [1, 0, 1, 0]
p_a = [0.9, 0.1, 0.8, 0.3]
p_b = [0.6, 0.4, 0.6, 0.4]

def log_loss(y_true, p_hat):
    n = len(y_true)
    return -sum(yi * math.log(pi) + (1 - yi) * math.log(1 - pi)
                for yi, pi in zip(y_true, p_hat)) / n

print(f"log-loss A = {log_loss(y, p_a):.3f}   log-loss B = {log_loss(y, p_b):.3f}")
# misma accuracy (4/4 con t=0.5), pero A da probabilidades mucho mejores


In [ ]:
# Ejercicio 3 — umbral por costos
c_fp, c_fn = 10, 490
t_estrella = c_fp / (c_fp + c_fn)
p_hat = 0.05
print(f"t* = {t_estrella:.3f}")
print("con t=0.5 :", "revisar" if p_hat >= 0.5 else "dejar pasar")
print("con t*    :", "revisar" if p_hat >= t_estrella else "dejar pasar")
# La fórmula exige p̂ calibrada: el umbral óptimo se calcula sobre probabilidades reales.


## Reflexión

1. El laboratorio barre umbrales maximizando accuracy. Si el costo de un falso negativo
   fuera 9 veces el de un falso positivo, ¿qué umbral de probabilidad sería óptimo y por
   qué la accuracy dejaría de ser la métrica correcta para elegirlo?
2. Un modelo produce p̂ = 0.99 para un caso que resulta negativo. Calcula su contribución
   al log-loss y compárala con la de un p̂ = 0.6 equivocado. ¿Qué propiedad de la entropía
   cruzada ilustra la diferencia?
3. ¿Por qué "mover el umbral" nunca puede arreglar un modelo mal calibrado, y qué
   procedimiento sí lo haría sin reentrenar los coeficientes?
